In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!mkdir -p datasets
!cp -r /content/drive/MyDrive/clean_detect_dataset/clean_flower_detector_splits datasets/
!cp /content/drive/MyDrive/clean_detect_dataset/data.yaml datasets/

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO

## Перевірка: чи всі зображення читаються

In [ ]:
import os
import cv2

def check_images(dir_path):
    broken = []
    for root, _, files in os.walk(dir_path):
        for f in files:
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                img_path = os.path.join(root, f)
                img = cv2.imread(img_path)
                if img is None:
                    broken.append(img_path)
    return broken

broken = check_images("datasets/clean_flower_detector_splits/images")
print("Биті зображення:", broken)


## Перевірка парності image ↔ label

In [ ]:
from pathlib import Path

def check_pairs(root):
    missing_labels = []
    missing_images = []

    for split in ["train", "val", "test"]:
        img_dir = Path(root) / "images" / split
        lbl_dir = Path(root) / "labels" / split

        imgs = set([p.stem for p in img_dir.glob("*.jpg")])
        lbls = set([p.stem for p in lbl_dir.glob("*.txt")])

        for i in imgs - lbls:
            missing_labels.append(str(img_dir / (i + ".jpg")))

        for l in lbls - imgs:
            missing_images.append(str(lbl_dir / (l + ".txt")))

    return missing_labels, missing_images


missing_lbl, missing_img = check_pairs("datasets/clean_flower_detector_splits")

print("Фото без label:", missing_lbl)
print("Label без фото:", missing_img)


## Перевірка bounding boxes

In [ ]:
import os

def check_bboxes(root):
    errors = []

    for subdir, _, files in os.walk(root):
        for f in files:
            if f.endswith(".txt"):
                path = os.path.join(subdir, f)
                with open(path, "r") as file:
                    for line in file:
                        parts = line.strip().split()

                        if len(parts) != 5:
                            errors.append((path, "wrong_format", line))
                            continue

                        cls, xc, yc, w, h = parts
                        try:
                            xc, yc, w, h = map(float, (xc, yc, w, h))
                        except:
                            errors.append((path, "not_float", line))
                            continue

                        if not (0 <= xc <= 1):
                            errors.append((path, "xc_out_of_range", line))
                        if not (0 <= yc <= 1):
                            errors.append((path, "yc_out_of_range", line))
                        if not (0 <= w <= 1):
                            errors.append((path, "w_out_of_range", line))
                        if not (0 <= h <= 1):
                            errors.append((path, "h_out_of_range", line))

                        if w <= 0 or h <= 0:
                            errors.append((path, "zero_size", line))

    return errors


errors = check_bboxes("datasets/clean_flower_detector_splits/labels")
print("BoundingBox errors:", errors[:20])
print("TOTAL ERRORS:", len(errors))


In [ ]:
import os

def fix_polygon_labels(root):
    fixed_files = []

    for subdir, _, files in os.walk(root):
        for f in files:
            if f.endswith(".txt"):
                path = os.path.join(subdir, f)

                with open(path, "r") as file:
                    lines = file.read().strip().split("\n")

                new_lines = []
                changed = False

                for line in lines:
                    parts = line.strip().split()

                    # Якщо НЕ 5 елементів → полігон
                    if len(parts) > 5:
                        changed = True
                        cls = parts[0]
                        coords = list(map(float, parts[1:]))

                        xs = coords[0::2]
                        ys = coords[1::2]

                        xmin = min(xs)
                        xmax = max(xs)
                        ymin = min(ys)
                        ymax = max(ys)

                        # Перетворення у YOLO формат
                        xc = (xmin + xmax) / 2
                        yc = (ymin + ymax) / 2
                        w = xmax - xmin
                        h = ymax - ymin

                        new_line = f"{cls} {xc} {yc} {w} {h}"
                        new_lines.append(new_line)
                    else:
                        new_lines.append(line)

                if changed:
                    with open(path, "w") as f_out:
                        f_out.write("\n".join(new_lines))
                    fixed_files.append(path)

    return fixed_files


fixed = fix_polygon_labels("datasets/clean_flower_detector_splits/labels")
print("Виправлені файли:", fixed)


In [ ]:
errors = check_bboxes("datasets/clean_flower_detector_splits/labels")
print("BoundingBox errors:", errors[:20])
print("TOTAL ERRORS:", len(errors))


In [ ]:
print(open("datasets/data.yaml").read())


# ТРЕНУВАННЯ

In [ ]:
!pip install -q ultralytics

from ultralytics import YOLO

In [ ]:
model = YOLO("yolo11l.pt")

In [ ]:
model.train(
    data="/content/datasets/data.yaml",
    imgsz=640,
    epochs=80,
    batch=16,
    optimizer="AdamW",
    lr0=0.0005,

    hsv_h=0.015,
    hsv_s=0.8,
    hsv_v=0.8,
    degrees=10,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    fliplr=0.7,
    flipud=0.0,
    mosaic=1.0,
    mixup=0.0,
)


## Метрики з тренування

In [ ]:
from IPython.display import Image, display

print("=== METRICS (mAP, Precision, Recall) ===")
display(Image(filename='runs/detect/train2/results.png'))


## Precision / Recall / F1 / mAP50 / mAP50-95 графіки

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

csv_path = "/content/runs/detect/train2/results.csv"
df = pd.read_csv(csv_path)

precision = df["metrics/precision(B)"]
recall = df["metrics/recall(B)"]
f1 = 2 * (precision * recall) / (precision + recall)

plt.figure(figsize=(28, 5))
titles = [
    "Precision",
    "Recall",
    "F1-score",
    "mAP50",
    "mAP50-95"
]
metrics = [
    precision,
    recall,
    f1,
    df["metrics/mAP50(B)"],
    df["metrics/mAP50-95(B)"]
]

for i, (title, metric) in enumerate(zip(titles, metrics)):
    plt.subplot(1, 5, i+1)
    plt.plot(df["epoch"], metric)
    plt.title(title)
    plt.xlabel("Epoch")
    plt.grid(True)

plt.tight_layout()
plt.show()
